# Data Preservation and Transformation Validation

## Objective

Verify that the original NSL-KDD and UNSW-NB15 datasets remain unchanged during analytical preprocessing.

Checks include:
- Original dataset dimensions
- Correct binary traffic-class mapping
- Preservation of original feature values
- No unexpected missing values
- Separation of training and testing partitions

In [1]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_dir = current_dir.parent
else:
    project_dir = current_dir

print("Project directory:", project_dir)

Project directory: C:\Users\fatim\Documents\network-intrusion-eda


In [2]:
nsl_columns = [
    "duration", "protocol_type", "service", "flag",
    "src_bytes", "dst_bytes", "land", "wrong_fragment",
    "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "attack_type", "difficulty_level"
]

nsl_train = pd.read_csv(
    project_dir / "data/raw/nsl_kdd/KDDTrain+.txt",
    header=None,
    names=nsl_columns
)

nsl_test = pd.read_csv(
    project_dir / "data/raw/nsl_kdd/KDDTest+.txt",
    header=None,
    names=nsl_columns
)

unsw_train = pd.read_csv(
    project_dir / "data/raw/unsw_nb15/UNSW_NB15_training-set.csv"
)

unsw_test = pd.read_csv(
    project_dir / "data/raw/unsw_nb15/UNSW_NB15_testing-set.csv"
)

original_datasets = {
    "NSL-KDD Train": nsl_train,
    "NSL-KDD Test": nsl_test,
    "UNSW-NB15 Train": unsw_train,
    "UNSW-NB15 Test": unsw_test
}

for name, df in original_datasets.items():
    print(name, df.shape)

NSL-KDD Train (125973, 43)
NSL-KDD Test (22544, 43)
UNSW-NB15 Train (175341, 45)
UNSW-NB15 Test (82332, 45)


In [3]:
analysis_datasets = {
    name: df.copy(deep=True)
    for name, df in original_datasets.items()
}

print("Analytical copies created successfully.")

Analytical copies created successfully.


In [4]:
for name, df in analysis_datasets.items():

    if name.startswith("NSL-KDD"):

        df["traffic_class"] = df["attack_type"].apply(
            lambda x: "Normal" if x == "normal" else "Attack"
        )

    else:

        df["traffic_class"] = df["label"].map({
            0: "Normal",
            1: "Attack"
        })

print("Traffic classes created successfully.")

Traffic classes created successfully.


In [5]:
validation_results = []

for name in original_datasets:

    original = original_datasets[name]
    analysis = analysis_datasets[name]

    same_rows = len(original) == len(analysis)

    original_columns_preserved = original.equals(
        analysis[original.columns]
    )

    class_values_valid = analysis[
        "traffic_class"
    ].isin(["Normal", "Attack"]).all()

    no_new_missing = (
        analysis.isna().sum().sum()
        ==
        original.isna().sum().sum()
    )

    validation_results.append({
        "Dataset": name,
        "Rows Preserved": same_rows,
        "Original Values Preserved": original_columns_preserved,
        "Traffic Classes Valid": class_values_valid,
        "No New Missing Values": no_new_missing
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

,Dataset,Rows Preserved,Original Values Preserved,Traffic Classes Valid,No New Missing Values
0,NSL-KDD Train,True,True,True,True
1,NSL-KDD Test,True,True,True,True
2,UNSW-NB15 Train,True,True,True,True
3,UNSW-NB15 Test,True,True,True,True


In [6]:
for name, df in analysis_datasets.items():

    print(f"\n{name}")

    print(
        df["traffic_class"]
        .value_counts()
    )


NSL-KDD Train
traffic_class
Normal    67343
Attack    58630
Name: count, dtype: int64

NSL-KDD Test
traffic_class
Attack    12833
Normal     9711
Name: count, dtype: int64

UNSW-NB15 Train
traffic_class
Attack    119341
Normal     56000
Name: count, dtype: int64

UNSW-NB15 Test
traffic_class
Attack    45332
Normal    37000
Name: count, dtype: int64


In [7]:
check_columns = [
    "Rows Preserved",
    "Original Values Preserved",
    "Traffic Classes Valid",
    "No New Missing Values"
]

assert validation_df[check_columns].all().all(), (
    "Validation failed! Review the results."
)

print("SUCCESS: All data preservation checks passed!")

SUCCESS: All data preservation checks passed!


In [8]:
output_path = (
    project_dir
    / "reports"
    / "data_validation_summary.csv"
)

validation_df.to_csv(
    output_path,
    index=False
)

print("Validation report saved:", output_path)
print("File exists:", output_path.exists())

Validation report saved: C:\Users\fatim\Documents\network-intrusion-eda\reports\data_validation_summary.csv
File exists: True
